# Figure 4: LLM Empathic Response Binary Classification Performance

This notebook generates Figure 4. It uses only aggregate F1 scores and confidence intervals and contains no patient-level data or PHI.

In [ ]:
# Figure 4: LLM Empathic Response Binary Classification Performance
# This notebook generates the accessibility-focused Figure 4 for the JAMIA manuscript.
# It uses only aggregate F1 scores and confidence intervals and contains no patient-level data or PHI.

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib import font_manager
import numpy as np
from pathlib import Path

definition_labels = ["No Definition", "Basic Definition", "Codebook", "Simplified\nCodebook"]
x = np.arange(len(definition_labels))

data = {
    "Llama-3.1-8B": {
        "Zero-shot": {
            "f1": np.array([0.52, 0.24, 0.69, 0.74]),
            "low": np.array([0.381, 0.0939, 0.5405, 0.6102]),
            "high": np.array([0.6706, 0.4398, 0.8191, 0.8553]),
        },
        "Few-shot": {
            "f1": np.array([0.65, 0.66, 0.63, 0.74]),
            "low": np.array([0.4961, 0.5163, 0.4877, 0.5862]),
            "high": np.array([0.7761, 0.7844, 0.7397, 0.8484]),
        },
    },
    "Llama-3.1-70B": {
        "Zero-shot": {
            "f1": np.array([0.79, 0.73, 0.76, 0.78]),
            "low": np.array([0.6549, 0.5956, 0.6269, 0.6421]),
            "high": np.array([0.8793, 0.8571, 0.8642, 0.8712]),
        },
        "Few-shot": {
            "f1": np.array([0.73, 0.75, 0.69, 0.77]),
            "low": np.array([0.6187, 0.6382, 0.5671, 0.6547]),
            "high": np.array([0.8250, 0.8375, 0.8000, 0.8644]),
        },
    },
}

for model, shots in data.items():
    for shot, vals in shots.items():
        for i, (f, lo, hi) in enumerate(zip(vals["f1"], vals["low"], vals["high"])):
            assert lo <= f <= hi, (model, shot, definition_labels[i], f, lo, hi)

def yerr_from_bounds(f1, low, high):
    return np.vstack([f1 - low, high - f1])

available_fonts = {f.name for f in font_manager.fontManager.ttflist}
font_name = "Open Sans" if "Open Sans" in available_fonts else "DejaVu Sans"

plt.rcParams.update({
    "font.family": font_name,
    "font.size": 10.8,
    "axes.labelsize": 11.5,
    "xtick.labelsize": 9.8,
    "ytick.labelsize": 10,
    "legend.fontsize": 10.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

zero_color = "#0072B2"
few_color = "#D55E00"
text_color = "#111111"
grid_color = "#CFCFCF"
spine_color = "#777777"
color_lookup = {"Zero-shot": zero_color, "Few-shot": few_color}
marker_lookup = {"Zero-shot": "o", "Few-shot": "s"}
offset_lookup = {"Zero-shot": -0.12, "Few-shot": 0.12}

fig = plt.figure(figsize=(12, 5.8))
gs = fig.add_gridspec(
    nrows=1, ncols=2,
    left=0.19, right=0.985, top=0.77, bottom=0.18,
    wspace=0.10
)
axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]

for ax, (model, shots) in zip(axes, data.items()):
    for shot, vals in shots.items():
        xpos = x + offset_lookup[shot]
        ax.errorbar(
            xpos, vals["f1"],
            yerr=yerr_from_bounds(vals["f1"], vals["low"], vals["high"]),
            fmt="none",
            ecolor=color_lookup[shot],
            elinewidth=1.6,
            capsize=0,
            linestyle="None",
            zorder=2
        )
    for shot, vals in shots.items():
        xpos = x + offset_lookup[shot]
        ax.plot(
            xpos, vals["f1"],
            marker=marker_lookup[shot],
            linestyle="None",
            markersize=6.5,
            markerfacecolor=color_lookup[shot],
            markeredgecolor="white",
            markeredgewidth=0.8,
            color=color_lookup[shot],
            label=shot,
            zorder=3
        )

    ax.set_ylim(0, 1.0)
    ax.set_xlim(-0.55, len(definition_labels) - 0.45)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.set_xticks(x)
    ax.set_xticklabels(definition_labels)
    ax.set_xlabel("Definition type", labelpad=10, fontweight="semibold")
    ax.yaxis.grid(True, color=grid_color, linewidth=0.75, linestyle="--", dashes=(2, 2), alpha=0.9)
    ax.xaxis.grid(False)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color(spine_color)
        ax.spines[spine].set_linewidth(0.8)
    ax.tick_params(colors=text_color)

    if ax is axes[0]:
        ax.set_ylabel("F1 score", fontweight="semibold")
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis="y", length=0)

for ax, model_label in zip(axes, data.keys()):
    bbox = ax.get_position()
    x_mid = (bbox.x0 + bbox.x1) / 2
    fig.text(x_mid, 0.815, model_label, ha="center", va="center",
             fontsize=14.5, fontweight="semibold", color=text_color)
    fig.lines.append(plt.Line2D([bbox.x0, bbox.x1], [0.785, 0.785],
                                transform=fig.transFigure, color=text_color, linewidth=1.35))

bbox0 = axes[0].get_position()
y_mid = (bbox0.y0 + bbox0.y1) / 2
fig.text(0.07, y_mid, "Empathy", ha="left", va="center",
         fontsize=12.5, fontweight="semibold", color=text_color)

legend_y = 0.925
fig.text(0.43, legend_y, "PROMPT EXAMPLES:", ha="right", va="center",
         fontsize=11.5, fontweight="bold", color=text_color)

legend_handles = [
    Line2D([0], [0], marker="o", linestyle="None", color=zero_color,
           markerfacecolor=zero_color, markeredgecolor=zero_color,
           markersize=9, label="Zero-shot"),
    Line2D([0], [0], marker="s", linestyle="None", color=few_color,
           markerfacecolor=few_color, markeredgecolor=few_color,
           markersize=9, label="Few-shot"),
]
fig.legend(handles=legend_handles, loc="center", bbox_to_anchor=(0.59, legend_y),
           ncol=2, frameon=False, handletextpad=0.5, columnspacing=1.8, borderaxespad=0)

outdir = Path("figures")
outdir.mkdir(exist_ok=True)
fig.savefig(outdir / "Figure4.png", dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(outdir / "Figure4.pdf", bbox_inches="tight", facecolor="white")
fig.savefig(outdir / "Figure4.svg", bbox_inches="tight", facecolor="white")
plt.close(fig)

print(f"Font used: {font_name}")
